** Preparación de los datos**

In [3]:
import pandas as pd

#  Cargar el archivo
df = pd.read_csv('/content/LogsDataiku.csv.gz')


**N.º de visitas → Número de visitas (sesiones que ha tenido el sitio web)**

In [5]:
num_visitas = df['session_id'].nunique()

print(f"N.º de visitas (sesiones): {num_visitas}")

N.º de visitas (sesiones): 3946


**N.º de visitantes únicos → Número de usuarios diferentes que han visitado el sitio web**:


In [2]:
# 2. Contar los IDs de visitantes únicos
num_visitantes = df['visitor_id'].nunique()

print(f"N.º de visitantes únicos: {num_visitantes}")

N.º de visitantes únicos: 2537


N.º medio de páginas/visitas →Para cada visita (sesión) cuántas páginas se han visitado. Media
para todas las visitas

In [6]:
total_paginas = len(df)
num_visitas = df['session_id'].nunique()

# 3. Calcular la media
media_paginas = total_paginas / num_visitas

print(f"N.º medio de páginas/visitas: {media_paginas:.2f}")

N.º medio de páginas/visitas: 2.75


**Tasa de rebote → Número de visitas (sesiones) que solo han accedido a una página**

In [7]:
# 2. Contar cuántas páginas hay por cada sesión
vistas_por_sesion = df.groupby('session_id').size()

# 3. Filtrar las que tienen solo 1 página y contar el total de visitas
num_rebotes = (vistas_por_sesion == 1).sum()
total_visitas = df['session_id'].nunique()

# 4. Calcular el porcentaje
tasa_rebote = (num_rebotes / total_visitas) * 100

print(f"N.º de rebotes: {num_rebotes}")
print(f"Tasa de rebote: {tasa_rebote:.2f}%")

N.º de rebotes: 2165
Tasa de rebote: 54.87%


**Tasa de salida para cada página → % de veces que cada página ha sido una página de salida:**

In [8]:
import pandas as pd

# 1. Asegurar orden cronológico
df['server_ts'] = pd.to_datetime(df['server_ts'])
df_ordenado = df.sort_values(['session_id', 'server_ts'])

# 2. Obtener la última página de cada sesión (la salida)
paginas_salida = df_ordenado.groupby('session_id').tail(1)
conteo_salidas = paginas_salida['location'].value_counts()

# 3. Obtener vistas totales de cada página
vistas_totales = df['location'].value_counts()

# 4. Calcular la tasa
tasa_salida = (conteo_salidas / vistas_totales * 100).fillna(0)

# Ver resultados para las páginas principales
print(tasa_salida.head(10))

location
http://dataiku.com/                                  44.887781
http://dataiku.com/applications/                     17.687075
http://dataiku.com/applications/advertising/          0.000000
http://dataiku.com/applications/ecommerce/           29.166667
http://dataiku.com/applications/freemium/            15.686275
http://dataiku.com/applications/publishing/          16.666667
http://dataiku.com/applications/revenue_forecast/    17.391304
http://dataiku.com/applications/smartcities/          4.000000
http://dataiku.com/blog/                             38.461538
http://dataiku.com/blog/2012/07/                      0.000000
Name: count, dtype: float64


**Tráfico directo, Tráfico de búsqueda , Tráfico referido **



In [9]:
import pandas as pd

# 1. Obtener la primera página de cada sesión
df['server_ts'] = pd.to_datetime(df['server_ts'])
primer_acceso = df.sort_values(['session_id', 'server_ts']).groupby('session_id').head(1)

# 2. Definir función de clasificación
def categorizar(ref):
    if pd.isna(ref) or 'dataiku.com' in str(ref):
        return 'Directo'
    buscadores = ['google.', 'bing.', 'yahoo.', 'yandex.', 'duckduckgo.']
    if any(b in str(ref).lower() for b in buscadores):
        return 'Búsqueda'
    return 'Referido'

# 3. Aplicar y contar
primer_acceso['fuente'] = primer_acceso['referer'].apply(categorizar)
print(primer_acceso['fuente'].value_counts())

fuente
Directo     2265
Búsqueda     908
Referido     773
Name: count, dtype: int64
